In [1]:
import pandas as pd
import numpy as np
from langchain_openai import AzureChatOpenAI
import json
import re
from pathlib import Path
from datetime import datetime as dt

In [2]:
# 1. Acceleration Pedal (Acc): 138 lines with labels [1, 0, 0, 0, 0, 0].
# 2. Wheel Steering Angle (WSA): 160 lines with labels [0, 1, 0, 0, 0, 0].
# 3. Wheel Speed (WS): 104 lines with labels [0, 0, 1, 0, 0, 0].
# 4. Yaw Rate (YR): 107 lines with labels [0, 0, 0, 1, 0, 0].
# 5. Steering Torque (ST): 118 lines with labels [0, 0, 0, 0, 1, 0].
# 6. Vehicle Speed (VS): 102 lines with labels [0, 0, 0, 0, 0, 1].
# 7. Acceleration Pedal and Wheel Steering Angle (Acc and WSA): 142 lines with labels [1, 1, 0, 0, 0, 0].
# 8. Steering Angle and Wheel Speed (WSA and WS): 131 lines with labels [0, 1, 1, 0, 0, 0].

In [3]:
# read data and rename columns
df = pd.read_csv("data/requirements.csv")
df.columns = ["req", "s1", "s2", "s3", "s4", "s5", "s6"]

In [4]:
# drop vihecal speed sensor
df = df.loc[df["s6"]!=1]
# df.drop(columns="s6", inplace=True)

# Find Examples

In [5]:
# collect used examples to exclude from test dataset
indexes_to_drop = []

In [6]:
# collect examples with single fault
examples = {}

for c in df.columns[1:]:
    if df[c].sum() == 0: continue
    examples[c] = {}
    i = df.loc[ (df[c] == 1) & (df[df.columns[1:][df.columns[1:]!=c]].sum(axis=1) == 0) ].sample(1)
    indexes_to_drop.append(i.index)
    i = i.values[0]
    examples[c]["req"] = i[0]
    examples[c]["vec"] = "[" + ",".join(map(str, i[1:])) + "]"

In [7]:
# collect examples with multiple faults
examples_multiple = {}

t = df.loc[df[df.columns[1:]].sum(axis=1) == 2]
idx = t.drop(columns="req").drop_duplicates().index
indexes_to_drop.append(idx)
t = df.iloc[idx, :].values

for r in t:
    c = "&".join(df.columns[1:][r[1:]==1])
    examples_multiple[c] = {}
    examples_multiple[c]["req"] = r[0]
    examples_multiple[c]["vec"] = "[" + ",".join(map(str, r[1:])) + "]"

In [8]:
# add both single and multiple into one place
examples.update(examples_multiple)

In [9]:
# join all examples in a text format to add to prompt
examples_txt = ""

for e in examples.values():
    examples_txt += f"Requirement: {e['req']}\n"
    examples_txt += f"Vector: {e['vec']}\n"
    examples_txt += "\n"

In [10]:
print("\n".join(examples_txt.split("\n")[:5]))

Requirement: Upon detection of a sensor failure, the system must reduce engine power and notify the driver through a warning indicator
Vector: [1,0,0,0,0,0]

Requirement: The steering angle sensor must be calibrated correctly, and the system should verify alignment periodically or after a specified number of miles
Vector: [0,1,0,0,0,0]


In [11]:
# drop indexes used in examples
indexes_to_drop = np.concatenate(indexes_to_drop)
df.drop(index=indexes_to_drop, inplace=True)

# Agent

In [32]:
SystemPrompt="""ACT as a professional tester where you receive a requirement and output a vector of 0s and 1s.
The requirement describe a problem where you analyze it and identify the sensors that need to be testes to fulfill this requirement, then each index in the vector corresponds to a sensor.
So, you will switch the sensor to be tested to 1 in the vector and the other sensors simply leave them as 0s.
The length of the vector is the same as the number of sensors.

Here are the available sensors to know each one and their vectors to learn the appropriate index:
{sensors}

Here are examples on how the requirements and the proportionate vector looks like:
{examples}


Here is the requirement to be solved:
{requirement}

Make sure to split the requirement into parts, analyze each one, propose a list of targeted sensors, then explain your answer in less than 50 words.
Finally generate the vector output.

Think a step by step to fulfill the previous steps and think twice as some of the requirements are tricky.
Be careful, one requirement might target multiple sensors simultaneously, but not necessarily.

Your output should be in a json format as follow:
{{
    "targeted_sensors": [sensors 1, sensor 2, ...],
    "reasoning": "your reasoning here",
    "vector": [1,0, ...]
}}
If you do not know the answer or the requirement is not clear at all, output a vector full of 0, and keep your though of process before arriving to a dead end in the reasoning.
"""

In [13]:
from prompts.Sensors import Sensors

# LLM

In [33]:
# llm
llm = AzureChatOpenAI(
    deployment_name="gpt-35",
    temperature=0.0
)

In [15]:
# t = llm.invoke(f"Here are a list of sensors where each one used in a real time HIL simulator to inject a fault and test the system behavior. Describe each sensor in less than 50 words to give an general idea to the user what each one do. The sensors:\n{Sensors}")

In [16]:
df = pd.read_csv("data/incorrect-instances_gpt-35.csv")
df = df[['idx', 'requirement', 'true_vector']]

In [17]:
# select random instance
instance = df.sample(1).to_dict("records")[0]
idx = instance["idx"]
req = instance["requirement"]
vec = instance["true_vector"]

In [34]:
# system prompt
messages = [
    {'role': 'system',
     'content': SystemPrompt.format(sensors=Sensors,examples=examples_txt, requirement=req)}
]

In [35]:
# run LLM
response = llm.invoke(messages)
messages.append({"role":"assistant", "content":response.content})

In [41]:
instance

{'idx': 583,
 'requirement': 'The steering column must be inspected for structural integrity and the absence of excessive play at specified service intervals',
 'true_vector': '[0,0,0,0,1,0]'}

In [37]:
print(response.content)

{
    "targeted_sensors": ["Wheel Steering Angle (WSA)"],
    "reasoning": "The inspection of the steering column for structural integrity and excessive play requires monitoring the angle at which the steering wheel is turned, which is measured by the Wheel Steering Angle sensor.",
    "vector": [0,1,0,0,0,0]
}


In [38]:
vec

'[0,0,0,0,1,0]'

In [39]:
def parse_result(messages):
    '''Parse the LLM result'''
    j = json.loads(messages[-1]["content"])
    j = j['vector']
    return "[" + ",".join(map(str, j)) + "]"

parse_result(messages)

'[0,1,0,0,0,0]'

In [40]:
# is the result match the original
parse_result(messages) == vec

False

In [24]:
# number of tokens
response.response_metadata["token_usage"]

{'completion_tokens': 95, 'prompt_tokens': 890, 'total_tokens': 985}